In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 05. Inventory Optimization

## Objective

This notebook transforms the final demand forecast into inventory
decision-support outputs.

The forecasting model has already been developed and evaluated in the
previous stage.

The objective of this stage is to determine how forecasted demand can
support inventory decisions, including:

- expected demand,
- inventory requirements,
- reorder decisions,
- recommended inventory quantity.

The forecasting model remains fixed during this stage.

Business parameters used for optimization must be explicitly defined
and should not be assumed without supporting data.

In [2]:
import pandas as pd

RAW_PATH = "/content/drive/MyDrive/ForecastOpti/data/raw/retail_sales.csv"

raw_df = pd.read_csv(RAW_PATH)

print("Available columns:")
print(raw_df.columns.tolist())

print("\nDataset shape:")
print(raw_df.shape)

print("\nData types:")
print(raw_df.dtypes)

Available columns:
['date', 'store_id', 'item_id', 'sales', 'price', 'promo', 'weekday', 'month']

Dataset shape:
(4565000, 8)

Data types:
date         object
store_id     object
item_id      object
sales         int64
price       float64
promo         int64
weekday       int64
month         int64
dtype: object


## Optimization Approach

The available dataset contains historical sales, price, promotion,
weekday, and month information.

No explicit inventory, lead-time, ordering-cost, holding-cost, or
service-level variables are available in the current dataset.

Therefore, the optimization stage will initially focus on
forecast-driven inventory decision support rather than a full
cost-based inventory optimization model.

The recommendations will be derived from forecasted demand and
historical demand variability.

In [3]:
# Parameter yang tersedia dari dataset
available_parameters = [
    "sales",
    "price",
    "promo",
    "weekday",
    "month"
]

# Parameter inventory yang belum tersedia
unavailable_parameters = [
    "lead_time",
    "service_level",
    "holding_cost",
    "ordering_cost",
    "current_inventory"
]

print("Available optimization inputs:")
for parameter in available_parameters:
    print("-", parameter)

print("\nUnavailable business parameters:")
for parameter in unavailable_parameters:
    print("-", parameter)

Available optimization inputs:
- sales
- price
- promo
- weekday
- month

Unavailable business parameters:
- lead_time
- service_level
- holding_cost
- ordering_cost
- current_inventory


In [4]:
import pandas as pd

FORECAST_PATH = (
    "/content/drive/MyDrive/ForecastOpti/"
    "outputs/forecasts/test_forecast.csv"
)

forecast_df = pd.read_csv(FORECAST_PATH)

print("Forecast shape:", forecast_df.shape)

print("\nForecast columns:")
print(forecast_df.columns.tolist())

print("\nFirst rows:")
display(forecast_df.head())

Forecast shape: (460000, 5)

Forecast columns:
['date', 'store_id', 'item_id', 'actual_sales', 'predicted_sales']

First rows:


,date,store_id,item_id,actual_sales,predicted_sales
0,2023-07-01,store_1,item_1,33,23.764274
1,2023-07-02,store_1,item_1,44,24.710509
2,2023-07-03,store_1,item_1,47,28.430032
3,2023-07-04,store_1,item_1,55,32.241477
4,2023-07-05,store_1,item_1,58,33.495661


In [5]:
# Menghitung forecast error.
# Error positif berarti model melakukan underprediction.
# Error negatif berarti model melakukan overprediction.
forecast_df["forecast_error"] = (
    forecast_df["actual_sales"]
    - forecast_df["predicted_sales"]
)

# Menghitung absolute forecast error.
forecast_df["absolute_error"] = (
    forecast_df["forecast_error"].abs()
)

# Ringkasan error forecast.
error_summary = forecast_df[
    [
        "forecast_error",
        "absolute_error"
    ]
].describe()

display(error_summary)

,forecast_error,absolute_error
count,460000.000000,460000.000000
mean,3.088149,8.977142
std,10.628118,6.473355
min,-35.075144,0.000089
25%,-5.081171,3.930091
50%,2.811830,7.897014
75%,10.634857,12.667004
max,58.241611,58.241611


In [6]:
# Hanya mengambil error positif karena kita ingin mengukur
# seberapa besar model biasanya mengalami underprediction.
positive_errors = forecast_df.loc[
    forecast_df["forecast_error"] > 0,
    "forecast_error"
]

# Menggunakan persentil ke-90 sebagai demand buffer.
# Ini berarti buffer merepresentasikan tingkat underprediction
# yang relatif tinggi berdasarkan historical forecast errors.
demand_buffer = positive_errors.quantile(0.90)

print(f"Positive forecast errors : {len(positive_errors):,}")
print(f"90th percentile buffer   : {demand_buffer:.2f} units")

Positive forecast errors : 270,889
90th percentile buffer   : 19.86 units


In [7]:
# Menghitung ringkasan forecast per kombinasi store-item.
# Data actual hanya digunakan sebagai bagian dari evaluasi historis;
# rekomendasi didasarkan pada predicted_sales + demand buffer.

inventory_summary = (
    forecast_df
    .groupby(["store_id", "item_id"])
    .agg(
        avg_forecast_demand=("predicted_sales", "mean"),
        max_forecast_demand=("predicted_sales", "max"),
        avg_actual_demand=("actual_sales", "mean"),
        demand_std=("actual_sales", "std")
    )
    .reset_index()
)

# Demand buffer berasal dari hasil analisis forecast error
# yang sudah kita hitung sebelumnya.
inventory_summary["demand_buffer"] = demand_buffer

# Recommended stock level:
# rata-rata forecast + buffer underprediction.
inventory_summary["recommended_stock"] = (
    inventory_summary["avg_forecast_demand"]
    + inventory_summary["demand_buffer"]
)

# Membulatkan nilai agar lebih mudah dibaca sebagai unit inventory.
inventory_summary["recommended_stock"] = (
    inventory_summary["recommended_stock"].round().astype(int)
)

print("Inventory recommendation rows:", len(inventory_summary))

display(
    inventory_summary.sort_values(
        "recommended_stock",
        ascending=False
    ).head(20)
)

Inventory recommendation rows: 2500


,store_id,item_id,avg_forecast_demand,max_forecast_demand,avg_actual_demand,demand_std,demand_buffer,recommended_stock
2241,store_5,item_47,30.644813,51.075144,47.048913,12.548172,19.855809,51
1382,store_34,item_39,30.247237,46.864415,30.413043,7.646782,19.855809,50
717,store_22,item_25,30.604402,49.009656,40.880435,10.521217,19.855809,50
1054,store_29,item_13,30.540298,51.187662,51.092391,13.140982,19.855809,50
1091,store_29,item_47,30.632629,51.187662,49.836957,12.428838,19.855809,50
1568,store_38,item_26,30.305976,51.075144,25.336957,6.955404,19.855809,50
1303,store_33,item_12,30.589415,51.075144,39.635870,9.947568,19.855809,50
878,store_25,item_35,30.571935,46.864415,27.478261,6.922646,19.855809,50
553,store_2,item_12,30.447145,51.075144,39.527174,9.564239,19.855809,50
2307,store_6,item_16,30.431553,51.187662,9.646739,3.853916,19.855809,50


In [8]:
# Menentukan batas kategori berdasarkan distribusi
# recommended stock antar store-item.
low_threshold = inventory_summary["recommended_stock"].quantile(0.33)
high_threshold = inventory_summary["recommended_stock"].quantile(0.67)

def classify_demand(stock_level):
    if stock_level >= high_threshold:
        return "High"
    elif stock_level <= low_threshold:
        return "Low"
    else:
        return "Medium"

inventory_summary["demand_category"] = (
    inventory_summary["recommended_stock"]
    .apply(classify_demand)
)

# Ringkasan jumlah store-item pada setiap kategori.
category_summary = (
    inventory_summary["demand_category"]
    .value_counts()
    .rename_axis("category")
    .reset_index(name="store_item_count")
)

display(category_summary)

print("\nThresholds:")
print(f"Low threshold  : {low_threshold:.2f}")
print(f"High threshold : {high_threshold:.2f}")

,category,store_item_count
0,High,1152
1,Low,906
2,Medium,442



Thresholds:
Low threshold  : 42.00
High threshold : 44.00


In [9]:
# Memilih kolom utama untuk output optimization.
optimization_output = inventory_summary[
    [
        "store_id",
        "item_id",
        "avg_forecast_demand",
        "max_forecast_demand",
        "demand_std",
        "demand_buffer",
        "recommended_stock",
        "demand_category"
    ]
].copy()

# Mengurutkan dari recommended stock tertinggi.
# Ini membuat item dengan kebutuhan inventory terbesar
# muncul terlebih dahulu ketika output dibaca.
optimization_output = optimization_output.sort_values(
    "recommended_stock",
    ascending=False
).reset_index(drop=True)

print("Optimization output shape:", optimization_output.shape)

display(
    optimization_output.head(20)
)

Optimization output shape: (2500, 8)


,store_id,item_id,avg_forecast_demand,max_forecast_demand,demand_std,demand_buffer,recommended_stock,demand_category
0,store_5,item_47,30.644813,51.075144,12.548172,19.855809,51,High
1,store_34,item_39,30.247237,46.864415,7.646782,19.855809,50,High
2,store_22,item_25,30.604402,49.009656,10.521217,19.855809,50,High
3,store_29,item_13,30.540298,51.187662,13.140982,19.855809,50,High
4,store_29,item_47,30.632629,51.187662,12.428838,19.855809,50,High
5,store_38,item_26,30.305976,51.075144,6.955404,19.855809,50,High
6,store_33,item_12,30.589415,51.075144,9.947568,19.855809,50,High
7,store_25,item_35,30.571935,46.864415,6.922646,19.855809,50,High
8,store_2,item_12,30.447145,51.075144,9.564239,19.855809,50,High
9,store_6,item_16,30.431553,51.187662,3.853916,19.855809,50,High


In [10]:
import os

OPT_DIR = "/content/drive/MyDrive/ForecastOpti/outputs/optimization"
os.makedirs(OPT_DIR, exist_ok=True)

optimization_path = os.path.join(
    OPT_DIR,
    "inventory_recommendations.csv"
)

optimization_output.to_csv(
    optimization_path,
    index=False
)

print("Optimization output saved to:")
print(optimization_path)

Optimization output saved to:
/content/drive/MyDrive/ForecastOpti/outputs/optimization/inventory_recommendations.csv


In [11]:
%cd /content/drive/MyDrive/ForecastOpti
!streamlit run src/dashboard.py --server.address 0.0.0.0 --server.port 8501 > streamlit.log 2>&1 &

/content/drive/MyDrive/ForecastOpti


In [12]:
!sleep 3
!curl -I http://127.0.0.1:8501

curl: (7) Failed to connect to 127.0.0.1 port 8501 after 0 ms: Connection refused


In [13]:
%cd /content/drive/MyDrive/ForecastOpti

!pkill -f "streamlit run" || true

!streamlit run src/dashboard.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    > streamlit.log 2>&1 &

/content/drive/MyDrive/ForecastOpti
^C


In [14]:
!sleep 5
!cat streamlit.log

/bin/bash: line 1: streamlit: command not found


In [15]:
!pip install -q streamlit==1.61.1

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 105.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.0 MB/s eta 0:00:00


In [16]:
%cd /content/drive/MyDrive/ForecastOpti

!python -m streamlit run src/dashboard.py \
    --server.address 0.0.0.0 \
    --server.port 8501 \
    > streamlit.log 2>&1 &

/content/drive/MyDrive/ForecastOpti


In [17]:
!sleep 5
!cat streamlit.log



2026-08-10 20:26:46.592 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://8.231.159.53:8501



In [18]:
!curl -I http://127.0.0.1:8501

HTTP/1.1 200 OK
date: Mon, 10 Aug 2026 20:27:03 GMT
server: uvicorn
content-type: text/html; charset=utf-8
accept-ranges: bytes
content-length: 10951
last-modified: Mon, 10 Aug 2026 20:26:31 GMT
etag: "6201063f1a7a2ea3b17413a3d59bb741"
cache-control: no-cache



In [20]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.7.3) ...
Setting up cloudflared (2026.7.3) ...
Processing triggers for man-db (2.10.2-1) ...


In [21]:
!cloudflared --version

cloudflared version 2026.7.3 (built 2026-07-23-09:58 UTC)


In [22]:
!cloudflared tunnel --url http://127.0.0.1:8501

2026-08-10T20:28:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-08-10T20:28:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-08-10T20:28:34Z INF +--------------------------------------------------------------------------------------------+
2026-08-10T20:28:34Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-08-10T20:28:34Z INF |  https://dialogue-frederick-weeks-survivor.trycloudfla